In [1]:
import pandas as pd;
import numpy as np;
import geopandas as gp;
 

# DOWNLOAD FILE FIRST: https://syd1.digitaloceanspaces.com/duckgoesmeow/bushfire-data/data.csv
FILE_PATH = "~/Desktop/archive/data.csv"

bushfire_df = pd.read_csv(FILE_PATH, low_memory=False)

columns_of_interest = ['FOD_ID','DISCOVERY_DATE', 'DISCOVERY_TIME', 'DISCOVERY_DOY', 'NWCG_GENERAL_CAUSE', 'FIRE_SIZE', 'FIRE_SIZE_CLASS', 'LATITUDE' , 'LONGITUDE' , 'STATE' ]
renamed_columns = ["fire_id", "discovery_date", "discovery_time", "discovery_doy", "general_cause", "fire_size", "fire_class", "latitude", "longitude", "state"]

# Rename the columns for better conventions:
bushfire_df.rename(columns={
                         'FOD_ID':'fire_id',
                         'DISCOVERY_DATE':'discovery_date',
                         'DISCOVERY_DOY': 'discovery_doy',
                         'DISCOVERY_TIME' : 'discovery_time',
                         'NWCG_GENERAL_CAUSE':'general_cause',
                         'FIRE_SIZE' : 'fire_size',
                         'FIRE_SIZE_CLASS' : 'fire_class',
                         'LATITUDE': 'latitude',
                         'LONGITUDE':'longitude',
                         'STATE':'state'}, inplace=True)

In [2]:
# Remove unnecessary columns - keep what we need =)
bushfire_df = bushfire_df[renamed_columns]

In [3]:
# Create discovery month for easier exploration
bushfire_df.insert(loc=2, column='discovery_month', value=pd.to_datetime(bushfire_df['discovery_date']).dt.month, allow_duplicates=False)

In [4]:
# Cast discovery_date to allow for further reformating
bushfire_df['discovery_date'] = pd.to_datetime(arg=bushfire_df['discovery_date']).astype(str)

In [5]:
# Replace not-present discovery_time values and re-cast to string for further reformating
bushfire_df['discovery_time'] = pd.to_numeric(bushfire_df['discovery_time'], errors='coerce').fillna(0).astype(int).astype(str)

In [6]:
# Convert the default format for better conventions and readability
def format_time(time_str):
    if pd.isna(time_str):
        return np.nan
    time_str = str(time_str).zfill(4)
    if time_str == '0000' or time_str == '2400':
        return '00:00'
    elif 0 <= int(time_str) <= 2359:
        return f"{time_str[:2]}:{time_str[2:]}"
    else: 
        return np.nan
    
# invoke our 'format_time' function
bushfire_df['discovery_time'] = bushfire_df['discovery_time'].apply(format_time)

In [7]:
# NOTE:
# The 'origin' variable isn't really required for intial focus on Natural causes because -
# -the general_cause will be Natural by default. 
# However, if we have time available, we can modify our to include more other causes.


# Dictionary/Object to categorize the general_cause for better classication
map_cause = {'Power generation/transmission/distribution':'Accidental',
            'Natural':'Natural',
            'Debris and open burning':'Accidental',
            'Missing data/not specified/undetermined':'Missing',
            'Recreation and ceremony':'Accidental',
            'Equipment and vehicle use':'Accidental',
            'Arson/incendiarism':'Criminal',
            'Fireworks':'Accidental',
            'Other causes':'Accidental',
            'Railroad operations and maintenance':'Accidental',
            'Smoking':'Accidental',
            'Misuse of fire by a minor':'Accidental',
            'Firearms and explosives use':'Accidental'}

# create a category identifier variable 'origin' to better categorize the bushfire cause
bushfire_df.insert(loc=6, column='origin', value=bushfire_df['general_cause'].map(map_cause))

In [8]:
# create a 'season' variable for data analysis
def yieldCurrentSeason(month: int) -> str:
    if month in [12, 1, 2]:  # Winter
        return "winter"
    elif month in [3, 4, 5]:  # Spring
        return "spring"
    elif month in [6, 7, 8]:  # Summer
        return "summer"
    elif month in [9, 10, 11]:  # Autumn
        return "autumn"
    else:
        return "Invalid month"

# create 'season' variable and invoke 'yieldCurrentSeason'
bushfire_df.insert(loc=12, column='season', value=bushfire_df['discovery_month'].apply(yieldCurrentSeason))

In [9]:
# NOTE:
# For now, let's only focus on predicting natural causes and if time allows, we can tune our model to consider other human related origins causes.
bushfire_df.drop(bushfire_df.loc[bushfire_df['origin'] != 'Natural'].index, inplace=True)

In [10]:
output_file = './cleaned-bushfire-data.csv'
bushfire_df.to_csv(output_file, index=False, encoding='utf-8')

In [11]:
# TODO: Test all of the fire_size and see if the match with their corresponding fire_class
# fire_size_classification = [{"size_min": 0.1, "size_max": 0.25, "class": "A"}, {"size_min": 0.26, "size_max": 9, "class": "B"}, { "size_min": 9.1, "size_max": 99, "class": "C"}, { "size_min": 100, "size_max": 299, "class": "D"}, { "size_min": 300, "size_max": 999, "class": "E"}, { "size_min": 1000, "size_max": 4999, "class": "F"}, { "size_min": 5000, "size_max": 9999, "class": "G"}]

In [12]:
# isSpatialWithinBounds(cleaned_df['latitude'], cleaned_df['longitude'], "CA")

# cleaned_df.head()

# match by state first, then narrow it down by matching the lat and long
# draw graph map of most bushfire-dense states
# it's to uncover patterns with the bushfires
# TODO: Gather all of the tempretures within the state and see how much they vary between different 